In [ ]:
!pip install pandas sentence-transformers rapidfuzz

In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer, util
from rapidfuzz import process, fuzz
import torch

class ClinicalHarmonizer:
    def __init__(self):
        # Using SapBERT: The gold standard for clinical entity alignment
        self.model = SentenceTransformer('cambridgeltl/SapBERT-from-PubMedBERT-fulltext')
        
        # Mock Standard Codebook (In a real scenario, load RxNorm/SNOMED/LOINC)
        self.codebook = {
            "RxNorm": [
                {"code": "198440", "term": "Acetaminophen 500 MG Oral Tablet"},
                {"code": "161", "term": "Albuterol 2 MG/ML Inhalation Solution"},
                {"code": "313782", "term": "Amlodipine 5 MG Oral Tablet"}
            ],
            "SNOMED": [
                {"code": "267036007", "term": "Dyspnea (Shortness of breath)"},
                {"code": "38341003", "term": "Hypertension"},
                {"code": "195967001", "term": "Asthma"}
            ]
        }
        # Pre-calculate embeddings for the codebook
        self.target_terms = [item['term'] for system in self.codebook.values() for item in system]
        self.term_to_metadata = {item['term']: item for system in self.codebook.values() for item in system}
        self.codebook_embeddings = self.model.encode(self.target_terms, convert_to_tensor=True)

    def preprocess(self, text):
        """Clean and normalize the input string."""
        if not text: return ""
        text = text.lower().strip()
        # Basic mapping for common abbreviations
        abbrev_map = {"hb": "hemoglobin", "bp": "blood pressure", "sob": "shortness of breath"}
        return abbrev_map.get(text, text)

    def get_best_match(self, raw_input, threshold=0.7):
        cleaned_input = self.preprocess(raw_input)
        
        # 1. Semantic Search (Deep Learning)
        query_embedding = self.model.encode(cleaned_input, convert_to_tensor=True)
        cos_scores = util.cos_sim(query_embedding, self.codebook_embeddings)[0]
        
        top_results = torch.topk(cos_scores, k=1)
        score = top_results.values[0].item()
        best_term = self.target_terms[top_results.indices[0]]

        # 2. String Similarity Fallback (Fuzzy matching)
        # Useful for typos that embeddings might miss
        fuzzy_match = process.extractOne(cleaned_input, self.target_terms, scorer=fuzz.WRatio)
        
        # Hybrid logic: favor semantic if score is high, otherwise check fuzzy
        if score > threshold:
            match_data = self.term_to_metadata[best_term]
            return {**match_data, "confidence": round(score, 4), "method": "Semantic"}
        else:
            return {"code": "UNKNOWN", "term": "No high confidence match", "confidence": 0, "method": "None"}

# --- Execution ---
harmonizer = ClinicalHarmonizer()

# Sample messy data from the workshop
test_inputs = ["Paracetamol 500mg", "Shortness of breath", "Astma", "BP high"]

results = []
for entry in test_inputs:
    match = harmonizer.get_best_match(entry)
    results.append({"Raw Input": entry, **match})

df_output = pd.DataFrame(results)
print(df_output)